<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Ignacio Díaz
- Nombre de alumno 2: Benjamín Fuentes


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [1]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 126 packages in 16ms
 Downloaded pillow
 Downloaded llvmlite
 Downloaded numba
Prepared 6 packages in 1.46s
Uninstalled 1 package in 31ms
Installed 6 packages in 918ms
 + llvmlite==0.47.0
 + numba==0.65.1
 - pillow==12.2.0
 + pillow==12.1.1
 + pynndescent==0.6.0
 + tqdm==4.67.3
 + umap-learn==0.5.12


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [4]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    # Se calcula Length.
    length = (df.groupby("Customer ID")["InvoiceDate"].max() - df.groupby("Customer ID")["InvoiceDate"].min()).dt.days

    # Se calcula Recency.
    thresholdDate = df["InvoiceDate"].max() + pd.Timedelta(days=1)
    recency = (thresholdDate - df.groupby("Customer ID")["InvoiceDate"].max()).dt.days

    # Se calcula Frequency.
    frequency = df.groupby("Customer ID")["Invoice"].nunique()

    # Se calcula Monetary.
    df["Profit"] = df["Quantity"] * df["Price"]
    monetary = df.groupby("Customer ID")["Profit"].sum() / frequency

    # Se calcula Periodicity.
    df_unico = df.drop_duplicates(["Customer ID", "Invoice"]).copy()
    df_unico = df_unico.sort_values(by=["Customer ID", "InvoiceDate"])
    periodicity = df_unico.groupby("Customer ID")["InvoiceDate"].diff().dt.days.groupby(df_unico["Customer ID"]).std()

    # Se crea dataframe resultante.
    dataframe_out = pd.DataFrame(
        {"Length": length, "Recency": recency, "Frequency": frequency, "Monetary": monetary, "Periodicity": periodicity}
    )  # Se orden solos por el index Customer ID.

    return dataframe_out

## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [5]:
class MinMax(BaseEstimator, TransformerMixin):
    def __init__(self):

        # Se definen las variables donde se guardan los mínimos y máximos de cada variable.
        self._min = None
        self._max = None

    def fit(self, X):

        # Se calcula el mínimo y máximo de cada variable.
        self._min = X.min(axis=0)
        self._max = X.max(axis=0)

        return self

    def transform(self, X):
        newX = (X - self._min) / (self._max - self._min)

        return newX

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from umap import UMAP

c:\Users\ignac\MDS7202\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Características principales.
features = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]

# Se crea el transformer de la función custom_features.
custom_features_trans = FunctionTransformer(custom_features)

# Se crea el estandarizador numérico.
standardizer = Pipeline(
    [("Imputador", SimpleImputer(strategy="constant", fill_value=0)), ("Escalador", MinMax())]
)  # Se agrega un imputador para lidear con nan de Periodicity.

# Se crea el estandarizador.
columnEstandardizer = ColumnTransformer(transformers=[("Estandarizador", standardizer, features)])

# Se crea el preprocesamiento.
preprocessor = Pipeline(
    [("Extracción características", custom_features_trans), ("Estandarizador", columnEstandardizer)]
)


pipeline_pca = Pipeline(
    [("Preprocesmiento", preprocessor), ("Reducción de dimensionalidad", PCA(n_components=2, random_state=0))]
)

pipeline_tsne = Pipeline(
    [("Preprocesmiento", preprocessor), ("Reducción de dimensionalidad", TSNE(n_components=2, random_state=0))]
)

pipeline_umap = Pipeline(
    [("Preprocesmiento", preprocessor), ("Reducción de dimensionalidad", UMAP(n_components=2, random_state=0))]
)

In [8]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

c:\Users\ignac\MDS7202\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [9]:
# Código para calcular loadings.

pca = pipeline_pca["Reducción de dimensionalidad"]
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

print(f"Loadings de las 2 principales componentes: \n{loadings}")

fig = px.scatter(pca_proj, x=0, y=1)

for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0,
        ay=0,
        axref="x",
        ayref="y",
        x=loadings[i, 0],
        y=loadings[i, 1],
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top",
    )
    fig.add_annotation(
        x=loadings[i, 0],
        y=loadings[i, 1],
        ax=0,
        ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

Loadings de las 2 principales componentes: 
[[ 0.34310143  0.09221018]
 [-0.18474321  0.18279055]
 [ 0.01771229  0.00354877]
 [ 0.00181505 -0.00045471]
 [ 0.07786398  0.02658275]]


### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Los loadings en PCA son valores que representan hacia donde apunta cada una de las características originales dentro del plano de los componentes de PCA, es decir, indican hacia donde crecen dentro de las nueva dimensiones.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: A partir de los loadings es posible saber la correlación que existe entre cada componente y la características originales, es decir, se puede saber que variable predomina en cada componente.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: Tanto en la figura anterior como en los valores de los loading se puede observar que el componente 0 está dominado por Length, un poco por Periodicity y el negativo de Recency, mientras que para el componente 1 domina más Recency y el negativo de Length.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [10]:
from sklearn.cluster import KMeans  # noqa: F401

inertias = []
k_values = range(1, 21)

for k in k_values:
    pipeline_kmeans = Pipeline(
        [("Preprocesamiento", preprocessor), ("KMeans", KMeans(n_clusters=k, random_state=0, n_init=10))]
    )
    pipeline_kmeans.fit(df_retail)
    inertias.append(pipeline_kmeans["KMeans"].inertia_)

fig = px.line(
    x=list(k_values),
    y=inertias,
    markers=True,
    title="Método del Codo",
    labels={"x": "Número de clusters (k)", "y": "Inercia"},
)
fig.show()

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: Se escogeria k=5 o 6, ya que es donde la curva empieza a aplanarse. Antes de ese punto la inercia baja bastante con cada cluster que se agrega, pero a partir de k=5 la reducción es pequeña, por lo que no vale la pena agregar más clusters.

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: Más o menos. Se puede ver una zona de inflexión alrededor de k=5 o k=6, pero el codo no es muy claro porque la curva baja de forma suave y no hay un quiebre brusco. Esto hace que la elección no sea del todo obvia solo mirando el gráfico.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: Se podría haber usado el coeficiente de silueta, que mide qué tan bien separado está cada punto respecto a su propio cluster versus los demás. Este da un número concreto para comparar distintos valores de k de forma más objetiva que solo mirar el gráfico.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [19]:
# Aquí calcule K-Means
# Entrenamos K-Means con k=6
pipeline_kmeans_final = Pipeline(
    [("Preprocesamiento", preprocessor), ("KMeans", KMeans(n_clusters=6, random_state=0, n_init=10))]
)
pipeline_kmeans_final.fit(df_retail)
kmeans_labels = pipeline_kmeans_final["KMeans"].labels_
df_lrmfp = custom_features(df_retail)
df_lrmfp["Cluster"] = kmeans_labels

# Tabla de promedios por cluster con conteo
resumen = df_lrmfp.groupby("Cluster")[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].mean().round(1)
resumen["Count"] = df_lrmfp.groupby("Cluster").size()
print(resumen)

         Length  Recency  Frequency  Monetary  Periodicity  Count
Cluster                                                          
0         275.1     38.7        3.9     349.4        112.9    298
1          31.6    188.3        1.8     340.7         23.0    602
2         322.5     22.4       11.5     425.2         32.5    916
3          17.1     43.7        1.7     377.9         14.6   1157
4         179.8     65.2        4.4     367.9         37.1    884
5           5.8    306.5        1.3     354.1         18.5    457


In [17]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=6", colors=kmeans_labels)

In [18]:
# Aquí grafique el Heatmap
fig_heatmap = px.imshow(
    resumen[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]],
    text_auto=True,
    aspect="auto",
    title="Heatmap de Características Promedio por Cluster",
    labels={"x": "Característica", "y": "Cluster"},
)
fig_heatmap.show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: En t-SNE y UMAP la separación es mas o menos clara, se pueden distinguir grupos  definidos con formas distintas. En PCA la separación es menos nítida ya que hay clusters que se superponen bastante, lo que se explica porque el PCA es una proyección lineal y no logra capturar bien la estructura no lineal de los datos.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: Sí, especialmente en t-SNE y UMAP donde se ven grupos con formas y posiciones distintas. El heatmap también confirma que los clusters tienen perfiles diferenciados, por ejemplo el cluster 1 tiene una Recency muy alta (188.3) lo que lo distingue claramente del resto, y el cluster 2 tiene el Length y Monetary más altos.

- ¿Quedarían mejor más o menos clusters?

> Respuesta: Con k=5 o 6 el resultado es razonable, pero viendo las proyecciones de t-SNE y UMAP se podrían justificar más clusters ya que hay subgrupos dentro de algunas regiones que K-Means fusionó en uno solo. Usar menos clusters perdería información relevante sobre los distintos perfiles de clientes.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: Probablemente no es el óptimo para este dataset. K-Means asume clusters esféricos y de tamaño similar, pero en las proyecciones de t-SNE y UMAP se ven grupos con formas irregulares y alargadas que K-Means no captura bien. Algoritmos como DBSCAN o clustering jerárquico se adaptarían mejor a estas formas

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: C0: Clientes leales irregulares: llevan mucho tiempo comprando (Length=275) pero con baja frecuencia (3.9) y alta variabilidad entre visitas (Periodicity=112.9). Compran esporádicamente pero llevan años en la tienda.
> C1: Clientes perdidos: Recency muy alta (188.3) y llevan poco tiempo como clientes (Length=31.6). Son clientes relativamente nuevos que dejaron de comprar rápido.
> C2: Clientes más valiosos: los que más tiempo llevan (Length=322.5), compran con más frecuencia (11.5) y generan el mayor ingreso promedio (Monetary=425.2). Son los mejores clientes de la tienda.
> C3: Clientes nuevos ocasionales: llevan muy poco tiempo (Length=17.1), compran poco (Frequency=1.7) y con baja Recency (43.7). Son clientes recientes que todavía no consolidan el hábito de compra.
> C4: Clientes intermedios en declive: llevan un tiempo moderado (Length=179.8) pero su Recency es relativamente alta (65.2), lo que indica que están comprando cada vez menos. Podrían estar perdiendo interés.
> C5: Clientes inactivos recientes: prácticamente no tienen historial (Length=5.8) y llevan muchísimo tiempo sin comprar (Recency=306.5). Probablemente hicieron una sola compra hace mucho y nunca volvieron.

Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [14]:
from sklearn.cluster import DBSCAN  # noqa: F401

# Aplicamos DBSCAN sobre la proyección UMAP (2D)
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(umap_proj)

print(f"Outliers detectados: {(dbscan_labels == -1).sum()}")
print(f"Clusters encontrados: {len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}")

Outliers detectados: 0
Clusters encontrados: 10


In [15]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: Se usó la proyección UMAP porque preserva bien la estructura local de los datos y hace que las regiones densas queden más compactas, facilitando que DBSCAN identifique zonas de alta y baja densidad. Aplicar DBSCAN directamente sobre las 5 dimensiones originales es más complicado porque al aumentar la dimensionalidad los conceptos de distancia y densidad se vuelven menos informativos (maldición de la dimensionalidad), haciendo muy difícil calibrar bien los parámetros.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: Se partió con eps=0.5 y min_samples=5 como valores de referencia estándar para datos 2D. Con estos parámetros DBSCAN encontró 10 clusters pero no detectó outliers, lo que indica que el eps utilizado fue suficientemente grande como para que todos los puntos tuvieran vecinos cercanos. Para detectar outliers habría que reducir el eps, lo que haría que puntos más aislados queden sin cluster asignado. El resultado igual es útil porque muestra que los datos no tienen muchos puntos verdaderamente aislados, sino que los clientes atípicos forman pequeños grupos entre sí.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: Dado que no se detectaron outliers con estos parámetros, se puede interpretar que no hay clientes completamente aislados en la proyección UMAP, sino que los más atípicos se agrupan con otros similares formando clusters pequeños. Mirando el gráfico, los clusters más pequeños y alejados del grupo principal (como los clusters 2, 4 o 5 en UMAP) probablemente corresponden a los clientes más inusuales, como mayoristas o cuentas corporativas con volúmenes de compra muy por encima del promedio. Estos coinciden con los perfiles extremos observados en el análisis LRMFP.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>